<a href="https://colab.research.google.com/github/th900/imovel_valor/blob/main/previsao_imovel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Previsão de Preços de Imóveis para Otimização de Margem Imobiliária

## 1. O Problema de Negócio (Contexto)
Em um mercado imobiliário altamente competitivo, a precificação correta de um imóvel é um dos fatores mais críticos para o sucesso de uma imobiliária digital.

* **A Dor do Negócio:** Atualmente, o processo de avaliação de imóveis é feito de forma manual por corretores. Isso gera dois grandes problemas:
  1. **Subprecificação:** Imóveis são avaliados abaixo do valor de mercado, fazendo com que o proprietário e a imobiliária percam dinheiro na comissão.
  2. **Superprecificação:** Imóveis são avaliados acima do valor real, ficando "encalhados" no site por meses, gerando custos de anúncio e frustração para os clientes.
  3. **Lentidão:** O processo manual demora dias para analisar todas as variáveis da casa (área, qualidade, ano de construção, etc.).

## 1.1 O que o Algoritmo Resolve?
O objetivo deste projeto é construir um modelo de Machine Learning capaz de prever o preço de venda de um imóvel (`SalePrice`) com base em suas características estruturais e de localização.

* **Impacto Operacional:** Reduzir o tempo de avaliação de dias para segundos.
* **Impacto Financeiro:** Maximizar o volume de vendas ao sugerir preços competitivos e justos, diminuindo o tempo de imóvel parado no catálogo.

### 2) Leitura de dados

In [4]:
import os
from google.colab import userdata

# O Colab busca o token com segurança sem expor seu texto no GitHub
os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')

In [3]:
# Baixa o zip do dataset oficial de preços de casas
!kaggle competitions download -c house-prices-advanced-regression-techniques

# Descompacta os arquivos .csv
!unzip house-prices-advanced-regression-techniques.zip

100% 199k/199k [00:00<00:00, 92.0MB/s]

Archive:  house-prices-advanced-regression-techniques.zip
  inflating: data_description.txt    
  inflating: sample_submission.csv   
  inflating: test.csv                
  inflating: train.csv               


In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [6]:
df_train = pd.read_csv('train.csv')
df_test = pd.read_csv('test.csv')

In [10]:
print(f"Quantidade de colunas do dataset de treino:{df_train.shape[1]} ")
print(f"Quantidade de linhas do dataset de treino: {df_train.shape[0]} ")
print(f"Quantidade de colunas do dataset de teste: {df_test.shape[1]} ")
print(f"Quantidade de linhas do dataset de teste: {df_test.shape[0]} ")

Quantidade de colunas do dataset de treino:81 
Quantidade de linhas do dataset de treino: 1460 
Quantidade de colunas do dataset de teste: 80 
Quantidade de linhas do dataset de teste: 1459 


In [8]:
df_train.head(5)

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [9]:
df_test.head(5)

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
0,1461,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,AllPub,...,120,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal
1,1462,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal
2,1463,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal
3,1464,60,RL,78.0,9978,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,6,2010,WD,Normal
4,1465,120,RL,43.0,5005,Pave,NaN,IR1,HLS,AllPub,...,144,0,NaN,NaN,NaN,0,1,2010,WD,Normal


### 3) Limpeza e tratamento dos dados

In [13]:
valores_nulos = df_train.isnull().sum()

colunas_com_nulos = valores_nulos[valores_nulos > 0].sort_values(ascending=False)

print(colunas_com_nulos)

PoolQC          1453
MiscFeature     1406
Alley           1369
Fence           1179
MasVnrType       872
FireplaceQu      690
LotFrontage      259
GarageType        81
GarageYrBlt       81
GarageFinish      81
GarageQual        81
GarageCond        81
BsmtExposure      38
BsmtFinType2      38
BsmtQual          37
BsmtCond          37
BsmtFinType1      37
MasVnrArea         8
Electrical         1
dtype: int64


In [14]:
# Criando uma cópia para preservar os dados originais
df_clean = df_train.copy()

# 1. Tratando colunas textuais onde nulo significa "Não possui"
text_cols_none = [
    'PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu',
    'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
    'BsmtExposure', 'BsmtFinType2', 'BsmtQual', 'BsmtCond', 'BsmtFinType1', 'MasVnrType'
]
for col in text_cols_none:
    df_clean[col] = df_clean[col].fillna('None')

# 2. Tratando colunas numéricas específicas
df_clean['MasVnrArea'] = df_clean['MasVnrArea'].fillna(0)
df_clean['GarageYrBlt'] = df_clean['GarageYrBlt'].fillna(0)

# 3. LotFrontage: Preenchendo com a mediana da frente dos lotes do mesmo bairro (Neighborhood)
df_clean['LotFrontage'] = df_clean.groupby('Neighborhood')['LotFrontage'].transform(lambda x: x.fillna(x.median()))

# 4. Electrical: Tem apenas 1 nulo, vamos preencher com a moda (o tipo mais comum)
df_clean['Electrical'] = df_clean['Electrical'].fillna(df_clean['Electrical'].mode()[0])

# Verificação final: Garante que o total de nulos zerou
print(f"Total de valores nulos restantes: {df_clean.isnull().sum().sum()}")

Total de valores nulos restantes: 0
